In [2]:
"""
2-D Numerical Modelling of Thermal Convection in the Earth's Mantle
Combined code for all three project members.

Person 1: Conduction model and validation
Person 2: Numerical convection simulation
Person 3: Nusselt number, plots, and sensitivity analysis

Install:
pip install numpy scipy matplotlib pandas
"""

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve


# ============================================================
# 1. SETTINGS
# ============================================================

RAYLEIGH_NUMBERS = [500, 1000, 2500, 5000]
GRID_SIZE = 41
END_TIME = 0.025
PERTURBATION_AMPLITUDE = 0.015

OUTPUT_BASE = Path("combined_mantle_results")

SENSITIVITY_GRIDS = [21, 31, 41]
SENSITIVITY_TIMES = [0.0125, 0.025]

DT_SAFETY = 0.20
MAX_STEPS = 200000


# ============================================================
# 2. OUTPUT FOLDER
# ============================================================

def make_output_folder():
    folder = OUTPUT_BASE
    counter = 2

    while folder.exists():
        folder = Path(f"{OUTPUT_BASE}_{counter}")
        counter += 1

    folder.mkdir(parents=True)
    return folder


# ============================================================
# 3. GRID
# ============================================================

def make_grid(n):
    x = np.linspace(0, 1, n)
    z = np.linspace(0, 1, n)

    X, Z = np.meshgrid(x, z)

    dx = x[1] - x[0]
    dz = z[1] - z[0]

    return x, z, X, Z, dx, dz


# ============================================================
# 4. NUMERICAL DERIVATIVES
# ============================================================

def laplacian(field, dx, dz):
    result = np.zeros_like(field)

    result[1:-1, 1:-1] = (
        (field[1:-1, 2:] - 2 * field[1:-1, 1:-1]
         + field[1:-1, :-2]) / dx**2
        +
        (field[2:, 1:-1] - 2 * field[1:-1, 1:-1]
         + field[:-2, 1:-1]) / dz**2
    )

    return result


def derivative_x(field, dx):
    return np.gradient(field, dx, axis=1, edge_order=2)


def derivative_z(field, dz):
    return np.gradient(field, dz, axis=0, edge_order=2)


# ============================================================
# 5. STREAMFUNCTION SOLVER
# ============================================================

def make_laplacian_matrix(n, dx, dz):
    interior_n = n - 2
    size = interior_n**2

    L = lil_matrix((size, size), dtype=float)

    def idx(i, j):
        return (i - 1) * interior_n + (j - 1)

    for i in range(1, n - 1):
        for j in range(1, n - 1):
            k = idx(i, j)

            L[k, k] = -2 / dx**2 - 2 / dz**2

            if j > 1:
                L[k, idx(i, j - 1)] = 1 / dx**2

            if j < n - 2:
                L[k, idx(i, j + 1)] = 1 / dx**2

            if i > 1:
                L[k, idx(i - 1, j)] = 1 / dz**2

            if i < n - 2:
                L[k, idx(i + 1, j)] = 1 / dz**2

    return L.tocsr()


def solve_streamfunction(rhs, dx, dz):
    """
    Simplified model:
        Laplacian(psi) = rhs

    This is an educational approximation, not the full
    biharmonic equation for mantle Stokes flow.
    """

    n = rhs.shape[0]
    L = make_laplacian_matrix(n, dx, dz)

    b = rhs[1:-1, 1:-1].ravel()
    solution = spsolve(L, b)

    psi = np.zeros_like(rhs)
    psi[1:-1, 1:-1] = solution.reshape((n - 2, n - 2))

    return psi


# ============================================================
# 6. TEMPERATURE INITIAL CONDITION AND BOUNDARIES
# ============================================================

def initial_temperature(X, Z):
    return (
        1 - Z
        + PERTURBATION_AMPLITUDE
        * np.sin(np.pi * X)
        * np.sin(np.pi * Z)
    )


def enforce_temperature_boundaries(T):
    # Hot bottom and cold top
    T[0, :] = 1.0
    T[-1, :] = 0.0

    # Approximate insulating sidewalls
    T[:, 0] = T[:, 1]
    T[:, -1] = T[:, -2]

    # Reapply top and bottom conditions
    T[0, :] = 1.0
    T[-1, :] = 0.0

    return T


# ============================================================
# 7. TIME STEP
# ============================================================

def stable_timestep(dx, dz, max_speed):
    diffusion_limit = DT_SAFETY / (
        2 * (1 / dx**2 + 1 / dz**2)
    )

    if max_speed <= 1e-12:
        advection_limit = diffusion_limit
    else:
        advection_limit = 0.35 / (
            max_speed * (1 / dx + 1 / dz)
        )

    return min(diffusion_limit, advection_limit)


# ============================================================
# 8. CONVECTION SIMULATION
# ============================================================

def run_convection(ra, n=GRID_SIZE, end_time=END_TIME):
    x, z, X, Z, dx, dz = make_grid(n)

    T = enforce_temperature_boundaries(
        initial_temperature(X, Z).copy()
    )

    t = 0.0
    steps = 0
    max_speed_seen = 0.0

    while t < end_time and steps < MAX_STEPS:

        dTdx = derivative_x(T, dx)

        psi = solve_streamfunction(
            -ra * dTdx, dx, dz
        )

        u = derivative_z(psi, dz)
        w = -derivative_x(psi, dx)

        speed = np.sqrt(u**2 + w**2)
        max_speed = np.max(speed[1:-1, 1:-1])

        max_speed_seen = max(max_speed_seen, max_speed)

        dt = min(
            stable_timestep(dx, dz, max_speed),
            end_time - t
        )

        if not np.isfinite(dt) or dt <= 0:
            raise RuntimeError(f"Invalid time step for Ra={ra}")

        # Upwind temperature derivatives
        dTdx_up = np.zeros_like(T)
        dTdz_up = np.zeros_like(T)

        dTdx_up[:, 1:-1] = np.where(
            u[:, 1:-1] >= 0,
            (T[:, 1:-1] - T[:, :-2]) / dx,
            (T[:, 2:] - T[:, 1:-1]) / dx
        )

        dTdz_up[1:-1, :] = np.where(
            w[1:-1, :] >= 0,
            (T[1:-1, :] - T[:-2, :]) / dz,
            (T[2:, :] - T[1:-1, :]) / dz
        )

        lap_T = laplacian(T, dx, dz)

        T_new = T.copy()

        T_new[1:-1, 1:-1] = (
            T[1:-1, 1:-1]
            + dt * (
                -u[1:-1, 1:-1]
                * dTdx_up[1:-1, 1:-1]
                -w[1:-1, 1:-1]
                * dTdz_up[1:-1, 1:-1]
                + lap_T[1:-1, 1:-1]
            )
        )

        T = enforce_temperature_boundaries(T_new)

        t += dt
        steps += 1

    # Final velocity field
    psi = solve_streamfunction(
        -ra * derivative_x(T, dx), dx, dz
    )

    u = derivative_z(psi, dz)
    w = -derivative_x(psi, dx)

    speed = np.sqrt(u**2 + w**2)

    divergence = (
        derivative_x(u, dx) + derivative_z(w, dz)
    )

    divergence_rms = np.sqrt(
        np.mean(divergence[1:-1, 1:-1]**2)
    )

    return {
        "ra": ra,
        "n": n,
        "end_time": end_time,
        "final_time": t,
        "steps": steps,
        "x": x,
        "z": z,
        "X": X,
        "Z": Z,
        "dx": dx,
        "dz": dz,
        "T": T,
        "psi": psi,
        "u": u,
        "w": w,
        "speed": speed,
        "max_speed": np.max(speed[1:-1, 1:-1]),
        "max_speed_seen": max_speed_seen,
        "divergence_rms": divergence_rms
    }


# ============================================================
# 9. NUSSELT NUMBER
# ============================================================

def calculate_nusselt(T, dx, dz):
    bottom_gradient = (
        T[1, 1:-1] - T[0, 1:-1]
    ) / dz

    top_gradient = (
        T[-1, 1:-1] - T[-2, 1:-1]
    ) / dz

    nu_bottom = -np.mean(bottom_gradient)
    nu_top = -np.mean(top_gradient)

    mismatch = (
        100 * abs(nu_bottom - nu_top)
        / max(abs(nu_bottom), abs(nu_top), 1e-12)
    )

    return nu_bottom, nu_top, mismatch


# ============================================================
# 10. CONDUCTION VALIDATION
# ============================================================

def conduction_validation(n=GRID_SIZE):
    x, z, X, Z, dx, dz = make_grid(n)

    T = 1 - Z
    lap_T = laplacian(T, dx, dz)

    nu_bottom, nu_top, mismatch = calculate_nusselt(
        T, dx, dz
    )

    return {
        "grid_n": n,
        "max_abs_laplacian": np.max(
            np.abs(lap_T[1:-1, 1:-1])
        ),
        "Nu_bottom": nu_bottom,
        "Nu_top": nu_top,
        "Nu_mismatch_percent": mismatch
    }


# ============================================================
# 11. SAVE RESULTS
# ============================================================

def save_run(run, output_dir):
    ra_label = str(run["ra"]).replace(".", "p")

    filename = output_dir / f"mantle_Ra_{ra_label}.npz"

    np.savez_compressed(
        filename,
        ra=run["ra"],
        n=run["n"],
        final_time=run["final_time"],
        steps=run["steps"],
        x=run["x"],
        z=run["z"],
        X=run["X"],
        Z=run["Z"],
        T=run["T"],
        psi=run["psi"],
        u=run["u"],
        w=run["w"],
        speed=run["speed"]
    )

    return filename


def summarize_run(run):
    nu_bottom, nu_top, mismatch = calculate_nusselt(
        run["T"], run["dx"], run["dz"]
    )

    return {
        "Ra": run["ra"],
        "grid_n": run["n"],
        "final_time": run["final_time"],
        "steps": run["steps"],
        "Nu_bottom": nu_bottom,
        "Nu_top": nu_top,
        "Nu_mismatch_percent": mismatch,
        "max_speed": run["max_speed"],
        "max_speed_seen": run["max_speed_seen"],
        "divergence_rms": run["divergence_rms"],
        "temperature_min": np.min(run["T"]),
        "temperature_max": np.max(run["T"]),
        "temperature_finite": np.isfinite(run["T"]).all(),
        "velocity_finite": (
            np.isfinite(run["u"]).all()
            and np.isfinite(run["w"]).all()
        )
    }


# ============================================================
# 12. CONDUCTION PLOTS
# ============================================================

def plot_conduction(output_dir):
    x, z, X, Z, dx, dz = make_grid(GRID_SIZE)

    T = 1 - Z

    fig, ax = plt.subplots(figsize=(7, 5))

    contour = ax.contourf(
        X, Z, T, levels=21, cmap="inferno"
    )

    fig.colorbar(contour, ax=ax, label="Dimensionless temperature")

    ax.set_xlabel("x / d")
    ax.set_ylabel("z / d")
    ax.set_title("Pure Conduction Temperature Field")
    ax.set_aspect("equal")

    fig.tight_layout()
    fig.savefig(output_dir / "conduction_temperature.png", dpi=180)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(6, 5))

    ax.plot(T[:, len(x)//2], z, label="T = 1 - z")
    ax.set_xlabel("Dimensionless temperature")
    ax.set_ylabel("z / d")
    ax.set_title("Conduction Temperature Profile")
    ax.grid(True, alpha=0.3)
    ax.legend()

    fig.tight_layout()
    fig.savefig(output_dir / "conduction_profile.png", dpi=180)
    plt.close(fig)


# ============================================================
# 13. PLOT EACH CONVECTION RUN
# ============================================================

def plot_run_fields(run, output_dir):
    X, Z = run["X"], run["Z"]
    ra_label = str(run["ra"]).replace(".", "p")

    # Temperature contours
    fig, ax = plt.subplots(figsize=(7, 5))

    contour = ax.contourf(
        X, Z, run["T"], levels=21, cmap="inferno"
    )

    fig.colorbar(contour, ax=ax, label="Dimensionless temperature")

    ax.set_xlabel("x / d")
    ax.set_ylabel("z / d")
    ax.set_title(f"Temperature Field, Ra = {run['ra']}")
    ax.set_aspect("equal")

    fig.tight_layout()
    fig.savefig(
        output_dir / f"temperature_Ra_{ra_label}.png",
        dpi=180
    )
    plt.close(fig)

    # Velocity magnitude and streamfunction
    fig, ax = plt.subplots(figsize=(7, 5))

    ax.contour(
        X, Z, run["psi"],
        levels=15, colors="black", linewidths=0.7
    )

    contour = ax.contourf(
        X, Z, run["speed"], levels=20, cmap="viridis"
    )

    fig.colorbar(
        contour, ax=ax,
        label="Dimensionless velocity magnitude"
    )

    ax.set_xlabel("x / d")
    ax.set_ylabel("z / d")
    ax.set_title(f"Velocity Field, Ra = {run['ra']}")
    ax.set_aspect("equal")

    fig.tight_layout()
    fig.savefig(
        output_dir / f"velocity_Ra_{ra_label}.png",
        dpi=180
    )
    plt.close(fig)


# ============================================================
# 14. COMPARISON PLOTS
# ============================================================

def plot_comparisons(runs, summary, output_dir):
    runs = sorted(runs, key=lambda item: item["ra"])

    # Maximum speed versus Rayleigh number
    fig, ax = plt.subplots(figsize=(7, 5))

    ax.plot(summary["Ra"], summary["max_speed"], "o-")

    ax.set_xlabel("Rayleigh number, Ra")
    ax.set_ylabel("Maximum dimensionless speed")
    ax.set_title("Maximum Speed vs Rayleigh Number")
    ax.grid(True, alpha=0.3)

    fig.tight_layout()
    fig.savefig(output_dir / "maximum_speed_vs_Ra.png", dpi=180)
    plt.close(fig)

    # Nusselt number versus Rayleigh number
    fig, ax = plt.subplots(figsize=(7, 5))

    ax.plot(
        summary["Ra"], summary["Nu_bottom"],
        "o-", label="Bottom Nu"
    )

    ax.plot(
        summary["Ra"], summary["Nu_top"],
        "s--", label="Top Nu"
    )

    ax.axhline(
        1, color="black", linestyle=":",
        label="Conduction reference Nu = 1"
    )

    ax.set_xlabel("Rayleigh number, Ra")
    ax.set_ylabel("Nusselt-number estimate")
    ax.set_title("Nusselt Number vs Rayleigh Number")
    ax.grid(True, alpha=0.3)
    ax.legend()

    fig.tight_layout()
    fig.savefig(output_dir / "nusselt_vs_Ra.png", dpi=180)
    plt.close(fig)

    # Log-log Nusselt plot
    positive = (
        (summary["Nu_bottom"] > 0)
        & np.isfinite(summary["Nu_bottom"])
    )

    fig, ax = plt.subplots(figsize=(7, 5))

    ax.loglog(
        summary.loc[positive, "Ra"],
        summary.loc[positive, "Nu_bottom"],
        "o-"
    )

    ax.set_xlabel("Rayleigh number, Ra")
    ax.set_ylabel("Bottom Nusselt estimate")
    ax.set_title("Exploratory Log-Log Nu vs Ra")
    ax.grid(True, which="both", alpha=0.3)

    fig.tight_layout()
    fig.savefig(output_dir / "loglog_nusselt_vs_Ra.png", dpi=180)
    plt.close(fig)

    # Vertical temperature profiles
    fig, ax = plt.subplots(figsize=(7, 5))

    for run in runs:
        mid_x = len(run["x"]) // 2

        ax.plot(
            run["T"][:, mid_x],
            run["z"],
            label=f"Ra = {run['ra']}"
        )

    z = runs[0]["z"]
    ax.plot(1 - z, z, "k--", label="Pure conduction")

    ax.set_xlabel("Dimensionless temperature")
    ax.set_ylabel("z / d")
    ax.set_title("Vertical Temperature Profiles at x = 0.5")
    ax.grid(True, alpha=0.3)
    ax.legend()

    fig.tight_layout()
    fig.savefig(
        output_dir / "vertical_temperature_profiles.png",
        dpi=180
    )
    plt.close(fig)

    # Velocity magnitude comparison
    fig, axes = plt.subplots(
        1, len(runs),
        figsize=(4 * len(runs), 4),
        squeeze=False,
        constrained_layout=True
    )

    vmax = max(np.max(run["speed"]) for run in runs)
    levels = np.linspace(0, max(vmax, 1e-12), 21)

    last_contour = None

    for ax, run in zip(axes[0], runs):
        last_contour = ax.contourf(
            run["X"], run["Z"], run["speed"],
            levels=levels, cmap="viridis", extend="max"
        )

        ax.set_title(f"Ra = {run['ra']}")
        ax.set_xlabel("x / d")
        ax.set_ylabel("z / d")
        ax.set_aspect("equal")

    fig.colorbar(
        last_contour,
        ax=axes.ravel().tolist(),
        label="Dimensionless velocity magnitude",
        shrink=0.85
    )

    fig.suptitle("Velocity Magnitude Comparison")
    fig.savefig(
        output_dir / "velocity_magnitude_comparison.png",
        dpi=180
    )
    plt.close(fig)


# ============================================================
# 15. GRID AND TIME SENSITIVITY
# ============================================================

def run_sensitivity_tests(output_dir):
    rows = []

    for n in SENSITIVITY_GRIDS:
        for end_time in SENSITIVITY_TIMES:

            print(
                f"Sensitivity test: grid={n}, time={end_time}"
            )

            run = run_convection(
                ra=2500,
                n=n,
                end_time=end_time
            )

            nu_bottom, nu_top, mismatch = calculate_nusselt(
                run["T"], run["dx"], run["dz"]
            )

            rows.append({
                "Ra": 2500,
                "grid_n": n,
                "end_time": end_time,
                "steps": run["steps"],
                "Nu_bottom": nu_bottom,
                "Nu_top": nu_top,
                "Nu_mismatch_percent": mismatch,
                "max_speed": run["max_speed"],
                "divergence_rms": run["divergence_rms"]
            })

    df = pd.DataFrame(rows)

    df.to_csv(
        output_dir / "grid_time_sensitivity.csv",
        index=False
    )

    fig, ax = plt.subplots(figsize=(7, 5))

    for end_time, group in df.groupby("end_time"):
        group = group.sort_values("grid_n")

        ax.plot(
            group["grid_n"], group["Nu_bottom"],
            "o-", label=f"Time = {end_time}"
        )

    ax.set_xlabel("Grid points per direction")
    ax.set_ylabel("Bottom Nusselt estimate")
    ax.set_title("Grid and Time Sensitivity")
    ax.grid(True, alpha=0.3)
    ax.legend()

    fig.tight_layout()
    fig.savefig(
        output_dir / "grid_time_sensitivity.png",
        dpi=180
    )
    plt.close(fig)

    return df


# ============================================================
# 16. MAIN PROGRAM: RUN ALL THREE MEMBERS' WORK
# ============================================================

def main():
    output_dir = make_output_folder()

    print("=" * 60)
    print("2-D MANTLE CONVECTION PROJECT")
    print("=" * 60)
    print("Results folder:", output_dir.resolve())

    # PERSON 1: CONDUCTION BASELINE
    print("\nPART 1: CONDUCTION VALIDATION")

    conduction = conduction_validation()

    pd.DataFrame([conduction]).to_csv(
        output_dir / "conduction_validation.csv",
        index=False
    )

    plot_conduction(output_dir)

    print("Conduction Nu bottom:", conduction["Nu_bottom"])
    print("Conduction Nu top:", conduction["Nu_top"])

    # PERSON 2: CONVECTION SIMULATIONS
    print("\nPART 2: CONVECTION SIMULATIONS")

    runs = []
    summaries = []

    for ra in RAYLEIGH_NUMBERS:
        print(f"\nRunning Ra = {ra}")

        run = run_convection(
            ra=ra,
            n=GRID_SIZE,
            end_time=END_TIME
        )

        runs.append(run)
        summaries.append(summarize_run(run))

        saved_file = save_run(run, output_dir)
        plot_run_fields(run, output_dir)

        print("Time steps:", run["steps"])
        print("Maximum speed:", run["max_speed"])
        print("Saved:", saved_file.name)

    summary = pd.DataFrame(summaries).sort_values("Ra")

    summary.to_csv(
        output_dir / "combined_summary.csv",
        index=False
    )

    # PERSON 3: HEAT TRANSPORT AND COMPARISON PLOTS
    print("\nPART 3: NUSSELT ANALYSIS AND COMPARISON PLOTS")

    plot_comparisons(runs, summary, output_dir)

    # Basic quality warnings
    warnings_list = []

    for _, row in summary.iterrows():
        if not row["temperature_finite"]:
            warnings_list.append(
                f"Non-finite temperature for Ra={row['Ra']}"
            )

        if not row["velocity_finite"]:
            warnings_list.append(
                f"Non-finite velocity for Ra={row['Ra']}"
            )

        if row["Nu_mismatch_percent"] > 10:
            warnings_list.append(
                f"Top/bottom Nu mismatch exceeds 10% "
                f"for Ra={row['Ra']}"
            )

        if (
            row["temperature_min"] < -0.05
            or row["temperature_max"] > 1.05
        ):
            warnings_list.append(
                f"Temperature outside approximate [0,1] "
                f"range for Ra={row['Ra']}"
            )

    # GRID AND TIME SENSITIVITY
    print("\nPART 4: GRID AND TIME SENSITIVITY")

    sensitivity = run_sensitivity_tests(output_dir)

    for end_time, group in sensitivity.groupby("end_time"):
        group = group.sort_values("grid_n")
        values = group["Nu_bottom"].to_numpy()

        if len(values) >= 2:
            relative_change = (
                abs(values[-1] - values[-2])
                / max(abs(values[-1]), 1e-12)
            )

            if relative_change > 0.10:
                warnings_list.append(
                    f"Nu changes by more than 10% between "
                    f"the two finest grids at time={end_time}"
                )

    # SAVE A TEXT REPORT
    report = [
        "MANTLE CONVECTION PROJECT REPORT",
        "=" * 40,
        f"Output directory: {output_dir.resolve()}",
        f"Rayleigh numbers: {RAYLEIGH_NUMBERS}",
        f"Grid size: {GRID_SIZE}",
        f"End time: {END_TIME}",
        "",
        "Scientific limitations:",
        "- Simplified educational convection model.",
        "- Streamfunction equation is approximate.",
        "- Nusselt estimates may be transient.",
        "- Sensitivity tests do not alone prove convergence.",
        "- Do not claim a validated Nu-Ra scaling law yet.",
        "",
        "Quality-control warnings:"
    ]

    if warnings_list:
        report.extend("- " + item for item in warnings_list)
    else:
        report.append("- No basic quality-control warnings triggered.")

    with open(
        output_dir / "run_report.txt",
        "w",
        encoding="utf-8"
    ) as file:
        file.write("\n".join(report))

    print("\n" + "=" * 60)
    print("ALL SIMULATIONS FINISHED")
    print("Results saved in:", output_dir.resolve())
    print("Check combined_summary.csv and run_report.txt.")
    print("=" * 60)


# ============================================================
# 17. RUN THE PROGRAM
# ============================================================

if __name__ == "__main__":
    main()

2-D MANTLE CONVECTION PROJECT
Results folder: C:\Users\athil\OneDrive\Desktop\combined_mantle_results_2

PART 1: CONDUCTION VALIDATION
Conduction Nu bottom: 1.0000000000000007
Conduction Nu top: 0.9999999999999971

PART 2: CONVECTION SIMULATIONS

Running Ra = 500
Time steps: 870
Maximum speed: 133.09757056314587
Saved: mantle_Ra_500.npz

Running Ra = 1000
Time steps: 1480
Maximum speed: 260.1929721500885
Saved: mantle_Ra_1000.npz

Running Ra = 2500
Time steps: 3340
Maximum speed: 488.3193460463304
Saved: mantle_Ra_2500.npz

Running Ra = 5000
Time steps: 6825
Maximum speed: 795.557041242009
Saved: mantle_Ra_5000.npz

PART 3: NUSSELT ANALYSIS AND COMPARISON PLOTS

PART 4: GRID AND TIME SENSITIVITY
Sensitivity test: grid=21, time=0.0125
Sensitivity test: grid=21, time=0.025
Sensitivity test: grid=31, time=0.0125
Sensitivity test: grid=31, time=0.025
Sensitivity test: grid=41, time=0.0125
Sensitivity test: grid=41, time=0.025

ALL SIMULATIONS FINISHED
Results saved in: C:\Users\athil\OneDr